# Genome Firewall — Clean, Safety-Aware AMR Model Pipeline

This notebook trains **one calibrated logistic-regression model per antibiotic** from processed bacterial genome features.

It keeps the organizer-provided split roles strictly separate:

- `train.csv`: model fitting only
- `calibration.csv`: probability calibration and no-call threshold selection only
- `hidden_test.csv`: final evaluation only

The notebook reports performance per antibiotic, including class-specific recall, PR-AUC, calibration quality, no-call coverage, and uncertainty intervals. It does **not** claim biological causation from model coefficients and does **not** claim sequence-homology isolation unless a genetic-group column is supplied.

> **Research prototype:** Every prediction must be confirmed with standard laboratory susceptibility testing.

In [1]:
# 1. Resolve paths and configure the run (Colab Drive or local repo checkout)
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Superbug-HackNation/processed_data')
    OUTPUT_DIR = Path('/content/drive/MyDrive/Superbug-HackNation/model_outputs_final')
except ImportError:
    # Running outside Colab (e.g. a local checkout of this repo): use the
    # processed_data/ and model_outputs_final/ directories that ship alongside
    # this notebook instead of a Drive mount.
    BASE_PATH = Path('processed_data')
    OUTPUT_DIR = Path('model_outputs_final')

SPLITS_PATH = BASE_PATH / 'splits'
TRAIN_PATH = SPLITS_PATH / 'train.csv'
CALIBRATION_PATH = SPLITS_PATH / 'calibration.csv'
TEST_PATH = SPLITS_PATH / 'hidden_test.csv'

# Reproducibility and policy settings
RANDOM_STATE = 42
N_BOOTSTRAPS = 500
TARGET_CALLED_ACCURACY = 0.90
MIN_CALIBRATION_COVERAGE = 0.30
MIN_CALLED_SAMPLES = 10
MIN_CLASS_COUNT_TRAIN = 2
MIN_CLASS_COUNT_CALIBRATION = 2
MIN_TEST_SAMPLES_FOR_DEPLOYMENT_FLAG = 20

# Descriptive post-evaluation flag only; never use hidden-test results to tune the model.
MIN_BALANCED_ACCURACY = 0.60
MIN_RESISTANT_RECALL = 0.50
MIN_DEPLOYMENT_COVERAGE = 0.30
MIN_DEPLOYMENT_CALLED_ACCURACY = 0.90

for directory in [
    OUTPUT_DIR,
    OUTPUT_DIR / 'models',
    OUTPUT_DIR / 'features',
    OUTPUT_DIR / 'thresholds',
    OUTPUT_DIR / 'metrics',
    OUTPUT_DIR / 'predictions',
    OUTPUT_DIR / 'plots' / 'reliability',
    OUTPUT_DIR / 'plots' / 'confusion_matrices',
    OUTPUT_DIR / 'plots' / 'roc',
    OUTPUT_DIR / 'plots' / 'precision_recall',
    OUTPUT_DIR / 'plots' / 'probability_distributions',
    OUTPUT_DIR / 'plots' / 'threshold_tradeoffs',
    OUTPUT_DIR / 'calibration_tables',
    OUTPUT_DIR / 'threshold_search',
    OUTPUT_DIR / 'explanations',
]:
    directory.mkdir(parents=True, exist_ok=True)

print('Input directory:', BASE_PATH)
print('Output directory:', OUTPUT_DIR)


Input directory: processed_data
Output directory: model_outputs_final


In [2]:
# 2. Imports and environment versions
import json
import math
import platform
import sys
import warnings
import zlib
from datetime import datetime, timezone

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

try:
    from sklearn.frozen import FrozenEstimator
    HAS_FROZEN_ESTIMATOR = True
except ImportError:
    FrozenEstimator = None
    HAS_FROZEN_ESTIMATOR = False

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 120)

print('Python:', sys.version)
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('scikit-learn:', sklearn.__version__)
print('joblib:', joblib.__version__)
print('FrozenEstimator available:', HAS_FROZEN_ESTIMATOR)

Python: 3.9.4 (v3.9.4:1f2e3088f3, Apr  4 2021, 12:32:44) 
[Clang 6.0 (clang-600.0.57)]
pandas: 2.3.3
numpy: 1.26.3
scikit-learn: 1.4.0
joblib: 1.5.3
FrozenEstimator available: False


## Data loading and validation

The code below checks the three supplied splits before any model is trained. Duplicate genome–antibiotic pairs are removed only when their complete rows are identical. Conflicting duplicates stop execution.

In [3]:
# 3. Load the fixed splits
REQUIRED_COLUMNS = {'genome_id', 'antibiotic', 'is_resistant'}
GROUP_COLUMN_CANDIDATES = [
    'genetic_group', 'cluster', 'cluster_id', 'lineage',
    'group', 'sequence_cluster'
]


def load_split(path: Path, split_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f'{split_name} file not found: {path}')
    df = pd.read_csv(path, dtype={'genome_id': 'string'})
    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f'{split_name} is missing required columns: {sorted(missing)}')
    df['genome_id'] = df['genome_id'].astype('string')
    df['antibiotic'] = df['antibiotic'].astype('string').str.strip()
    df['is_resistant'] = pd.to_numeric(df['is_resistant'], errors='raise')
    invalid_labels = sorted(set(df['is_resistant'].dropna().unique()) - {0, 1})
    if invalid_labels:
        raise ValueError(f'{split_name} has labels outside 0/1: {invalid_labels}')
    if df['is_resistant'].isna().any():
        raise ValueError(f'{split_name} contains missing labels.')
    df['is_resistant'] = df['is_resistant'].astype(int)
    return df


train_df = load_split(TRAIN_PATH, 'train')
calibration_df = load_split(CALIBRATION_PATH, 'calibration')
test_df = load_split(TEST_PATH, 'hidden_test')

print('Shapes:')
print('  train:', train_df.shape)
print('  calibration:', calibration_df.shape)
print('  hidden test:', test_df.shape)

display(train_df.head())

Shapes:
  train: (6699, 281)
  calibration: (2233, 281)
  hidden test: (2233, 281)


,genome_id,antibiotic,is_resistant,gene_aac(2')-Ic,gene_aac(3)-IId,gene_aac(3)-IIe,gene_aac(3)-IVa,gene_aac(3)-VIa,gene_aac(6')-Ib,gene_aac(6')-Ib',gene_aac(6')-Ib-cr5,gene_aac(6')-Ib3,gene_aac(6')-Ib4,gene_aadA1,gene_aadA16,gene_aadA2,gene_aadA25,gene_aadA5,gene_acrF,gene_acrR_E190RfsTer20,gene_acrR_E36Ter,gene_acrR_E67Ter,gene_acrR_E91KfsTer6,gene_acrR_E91Ter,gene_acrR_F52SfsTer3,gene_acrR_I112YfsTer19,gene_acrR_I70YfsTer2,gene_acrR_K116Ter,gene_acrR_K80Ter,gene_acrR_L89TerfsTer0,gene_acrR_M123WfsTer18,gene_acrR_P182RfsTer62,gene_acrR_P85HfsTer4,gene_acrR_Q14Ter,gene_acrR_Q27Ter,gene_acrR_Q78Ter,gene_acrR_T32RfsTer37,gene_acrR_V29GfsTer7,gene_acrR_V29YfsTer44,gene_acrR_V88IfsTer9,gene_acrR_W178Ter,gene_acrR_W50Ter,gene_ampC_Kaer-1,gene_ant(2'')-Ia,gene_aph(3'')-Ib,gene_aph(3')-Ia,gene_aph(3')-VI,gene_aph(4)-Ia,gene_aph(6)-Id,gene_arr,gene_arr-2,gene_arr-3,gene_blaC,gene_blaCARB-2,gene_blaCMY-2,gene_blaCMY-42,gene_blaCMY-6,gene_blaCMY-82,gene_blaCTX-M,gene_blaCTX-M-1,...,gene_tet(B),gene_tet(C),gene_tet(D),gene_tet(M),gene_tet(X4),gene_uhpT_E239RfsTer14,mut_acrR_G28R,mut_acrR_T5N,mut_ampC_C-11T,mut_ampC_C-42T,mut_ampC_T-32A,mut_blaTEMp_C32T,mut_blaTEMp_G162T,mut_cirA_Q42Ter,mut_cirA_S90YfsTer15,mut_cyaA_S352T,mut_fabI_F203L,mut_folP_P64S,mut_ftsI_G363S,mut_ftsI_N337NYRIN,mut_gyrA_D87G,mut_gyrA_D87N,mut_gyrA_D87Y,mut_gyrA_S83A,mut_gyrA_S83L,mut_marR_S3N,mut_nfsA_G126R,mut_nfsA_G131D,mut_nfsA_G154E,mut_nfsA_Q44Ter,mut_nfsA_Q67Ter,mut_nfsA_R15C,mut_nfsA_R203C,mut_nfsA_S33R,mut_nfsA_W159Ter,mut_nfsB_E197Ter,mut_ompC_Q82Ter,mut_ompF_Q88Ter,mut_parC_A108V,mut_parC_E84G,mut_parC_E84V,mut_parC_S57T,mut_parC_S80I,mut_parC_S80R,mut_parE_D475E,mut_parE_E460D,mut_parE_E460K,mut_parE_I355T,mut_parE_I529L,mut_parE_L416F,mut_parE_L445H,mut_parE_S458A,mut_pmrB_L14R,mut_pmrB_V161G,mut_ptsI_V25I,mut_rpsJ_V57L,mut_soxR_R20H,mut_soxS_A12S,mut_uhpT_E350Q,cluster_id
0,562.56783,ciprofloxacin,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1061
1,562.99772,meropenem,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2308
2,562.98600,ceftazidime,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,337
3,562.97941,ampicillin,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,28
4,562.100659,ceftazidime,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,237


In [4]:
# 4. Validate columns, duplicates, missing values, numeric features, and class balance

def resolve_group_column(dataframes):
    shared = set.intersection(*(set(df.columns) for df in dataframes))
    matches = [c for c in GROUP_COLUMN_CANDIDATES if c in shared]
    if len(matches) > 1:
        print('Multiple possible genetic-group columns found:', matches)
        print('Using:', matches[0])
    return matches[0] if matches else None


GROUP_COLUMN = resolve_group_column([train_df, calibration_df, test_df])
print('Genetic-group column:', GROUP_COLUMN)

# Require matching columns and preserve training-column order.
train_columns = list(train_df.columns)
for split_name, df in [('calibration', calibration_df), ('hidden_test', test_df)]:
    missing = set(train_columns) - set(df.columns)
    extra = set(df.columns) - set(train_columns)
    if missing or extra:
        raise ValueError(
            f'Column mismatch in {split_name}. Missing={sorted(missing)}, extra={sorted(extra)}'
        )
    df = df[train_columns]
    if split_name == 'calibration':
        calibration_df = df
    else:
        test_df = df

EXCLUDED_COLUMNS = ['genome_id', 'antibiotic', 'is_resistant']
if GROUP_COLUMN:
    EXCLUDED_COLUMNS.append(GROUP_COLUMN)
FEATURE_COLUMNS = [c for c in train_columns if c not in EXCLUDED_COLUMNS]

if not FEATURE_COLUMNS:
    raise ValueError('No model feature columns were found.')

# Convert all features to numeric. Fill missing values only for binary indicators.
missing_fill_report = []
non_binary_features = []
for split_name, df in [('train', train_df), ('calibration', calibration_df), ('hidden_test', test_df)]:
    for col in FEATURE_COLUMNS:
        df[col] = pd.to_numeric(df[col], errors='raise')
        finite_values = df[col].dropna().to_numpy(dtype=float)
        if finite_values.size and not np.isfinite(finite_values).all():
            raise ValueError(f'Infinite value found in {split_name}.{col}')
        unique_non_missing = set(df[col].dropna().unique().tolist())
        is_binary = unique_non_missing.issubset({0, 1, 0.0, 1.0})
        if not is_binary:
            non_binary_features.append((split_name, col, sorted(unique_non_missing)[:10]))
        missing_count = int(df[col].isna().sum())
        if missing_count:
            if is_binary:
                df[col] = df[col].fillna(0)
                missing_fill_report.append({
                    'split': split_name,
                    'feature': col,
                    'missing_filled_with_zero': missing_count,
                })
            else:
                raise ValueError(
                    f'Non-binary feature {col!r} in {split_name} contains missing values; '
                    'a documented imputation rule is required.'
                )

print(f'Feature count: {len(FEATURE_COLUMNS)}')
print(f'Non-binary feature observations: {len(non_binary_features)}')
if non_binary_features:
    display(pd.DataFrame(non_binary_features, columns=['split', 'feature', 'sample_values']).head(20))
if missing_fill_report:
    display(pd.DataFrame(missing_fill_report))


def deduplicate_identical_pairs(df: pd.DataFrame, split_name: str) -> tuple[pd.DataFrame, dict, pd.DataFrame]:
    # Two-stage cleanup: first collapse exact full-row duplicates, then check what
    # remains. A genome_id/antibiotic pair that still repeats after that with a
    # different label or feature vector is a genuine data conflict, not a duplicate
    # — we never guess which row is right, so every row for that pair is dropped
    # and logged rather than silently deduplicated.
    pair_cols = ['genome_id', 'antibiotic']
    duplicate_mask = df.duplicated(pair_cols, keep=False)
    duplicate_rows = df.loc[duplicate_mask]
    report = {
        'split': split_name,
        'duplicate_pair_rows': int(duplicate_mask.sum()),
        'duplicate_pairs': int(duplicate_rows[pair_cols].drop_duplicates().shape[0]),
        'exact_duplicate_rows_removed': 0,
        'conflicting_duplicate_pairs': 0,
        'conflicting_pairs_dropped_rows': 0,
    }
    if duplicate_rows.empty:
        return df.copy(), report, df.iloc[0:0].copy()

    before = len(df)
    cleaned = df.drop_duplicates(keep='first').copy()
    report['exact_duplicate_rows_removed'] = before - len(cleaned)

    remaining_mask = cleaned.duplicated(pair_cols, keep=False)
    remaining_duplicate_rows = cleaned.loc[remaining_mask]

    conflicting_pairs = []
    for pair, group in remaining_duplicate_rows.groupby(pair_cols, dropna=False):
        if len(group.drop_duplicates()) > 1:
            conflicting_pairs.append(pair)

    dropped_conflict_rows = cleaned.iloc[0:0].copy()
    if conflicting_pairs:
        conflict_mask = cleaned.set_index(pair_cols).index.isin(conflicting_pairs)
        dropped_conflict_rows = cleaned.loc[conflict_mask].copy()
        cleaned = cleaned.loc[~conflict_mask].copy()
        report['conflicting_duplicate_pairs'] = len(conflicting_pairs)
        report['conflicting_pairs_dropped_rows'] = int(conflict_mask.sum())
        print(
            f'WARNING: {split_name} had {len(conflicting_pairs)} genome_id/antibiotic pair(s) '
            f'with contradictory labels or feature values. All {int(conflict_mask.sum())} rows '
            f'for these pairs were dropped (not guessed): {conflicting_pairs}'
        )

    deduplicated = cleaned.drop_duplicates(pair_cols, keep='first').copy()
    report['rows_removed'] = int(len(df) - len(deduplicated) - report['conflicting_pairs_dropped_rows'])
    return deduplicated, report, dropped_conflict_rows


train_df, train_dup_report, train_dropped_conflicts = deduplicate_identical_pairs(train_df, 'train')
calibration_df, calibration_dup_report, calibration_dropped_conflicts = deduplicate_identical_pairs(calibration_df, 'calibration')
test_df, test_dup_report, test_dropped_conflicts = deduplicate_identical_pairs(test_df, 'hidden_test')

duplicate_report = pd.DataFrame([
    train_dup_report, calibration_dup_report, test_dup_report
])
display(duplicate_report)

dropped_conflicting_pairs_df = pd.concat(
    [train_dropped_conflicts, calibration_dropped_conflicts, test_dropped_conflicts],
    keys=['train', 'calibration', 'hidden_test'],
    names=['split'],
).reset_index(level=0).reset_index(drop=True)
if not dropped_conflicting_pairs_df.empty:
    (OUTPUT_DIR / 'metrics').mkdir(parents=True, exist_ok=True)
    dropped_conflicting_pairs_df.to_csv(
        OUTPUT_DIR / 'metrics' / 'dropped_conflicting_duplicate_pairs.csv', index=False
    )
    display(dropped_conflicting_pairs_df[['split', 'genome_id', 'antibiotic', 'is_resistant']])


def class_balance_table(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    counts = (
        df.groupby(['antibiotic', 'is_resistant'], observed=True)
          .size()
          .unstack(fill_value=0)
          .rename(columns={0: 'susceptible_count', 1: 'resistant_count'})
          .reset_index()
    )
    for col in ['susceptible_count', 'resistant_count']:
        if col not in counts:
            counts[col] = 0
    counts['n'] = counts['susceptible_count'] + counts['resistant_count']
    counts['resistant_prevalence'] = counts['resistant_count'] / counts['n']
    counts.insert(0, 'split', split_name)
    return counts

balance_table = pd.concat([
    class_balance_table(train_df, 'train'),
    class_balance_table(calibration_df, 'calibration'),
    class_balance_table(test_df, 'hidden_test'),
], ignore_index=True)

display(balance_table.sort_values(['antibiotic', 'split']))

Genetic-group column: cluster_id


Feature count: 277
Non-binary feature observations: 0


,split,duplicate_pair_rows,duplicate_pairs,exact_duplicate_rows_removed,conflicting_duplicate_pairs,conflicting_pairs_dropped_rows,rows_removed
0,train,6,3,0,3,6,0
1,calibration,2,1,0,1,2,0
2,hidden_test,2,1,0,1,2,0


,split,genome_id,antibiotic,is_resistant
0,train,562.58773,norfloxacin,0
1,train,562.58773,norfloxacin,1
2,train,562.58776,norfloxacin,1
3,train,562.58786,amoxicillin/clavulanic acid,0
4,train,562.58786,amoxicillin/clavulanic acid,1
5,train,562.58776,norfloxacin,0
6,calibration,562.65374,amoxicillin,1
7,calibration,562.65374,amoxicillin,0
8,hidden_test,562.65479,amoxicillin,0
9,hidden_test,562.65479,amoxicillin,1


is_resistant,split,antibiotic,susceptible_count,resistant_count,n,resistant_prevalence
59,calibration,amikacin,38,1,39,0.025641
110,hidden_test,amikacin,46,0,46,0.000000
0,train,amikacin,183,3,186,0.016129
60,calibration,amoxicillin,12,11,23,0.478261
111,hidden_test,amoxicillin,8,9,17,0.529412
...,...,...,...,...,...,...
56,train,trimethoprim,58,65,123,0.528455
109,calibration,trimethoprim/sulfamethoxazole,72,35,107,0.327103
161,hidden_test,trimethoprim/sulfamethoxazole,64,26,90,0.288889
57,train,trimethoprim/sulfamethoxazole,175,83,258,0.321705


In [5]:
# 5. Split-isolation and leakage checks

def overlap_count(left, right):
    return len(set(left) & set(right))

validation_report = {
    'exact_genome_id_overlap': {
        'train_calibration': overlap_count(train_df['genome_id'], calibration_df['genome_id']),
        'train_hidden_test': overlap_count(train_df['genome_id'], test_df['genome_id']),
        'calibration_hidden_test': overlap_count(calibration_df['genome_id'], test_df['genome_id']),
    },
    'genome_antibiotic_pair_overlap': {},
    'genetic_group_column': GROUP_COLUMN,
    'genetic_group_overlap': None,
}

pair_sets = {
    'train': set(map(tuple, train_df[['genome_id', 'antibiotic']].astype(str).to_numpy())),
    'calibration': set(map(tuple, calibration_df[['genome_id', 'antibiotic']].astype(str).to_numpy())),
    'hidden_test': set(map(tuple, test_df[['genome_id', 'antibiotic']].astype(str).to_numpy())),
}
for a, b in [('train', 'calibration'), ('train', 'hidden_test'), ('calibration', 'hidden_test')]:
    validation_report['genome_antibiotic_pair_overlap'][f'{a}_{b}'] = len(pair_sets[a] & pair_sets[b])

if any(validation_report['exact_genome_id_overlap'].values()):
    raise ValueError(f'Exact genome_id overlap detected: {validation_report["exact_genome_id_overlap"]}')
if any(validation_report['genome_antibiotic_pair_overlap'].values()):
    raise ValueError(
        'Genome–antibiotic pair overlap detected: '
        f'{validation_report["genome_antibiotic_pair_overlap"]}'
    )

if GROUP_COLUMN:
    group_sets = {
        'train': set(train_df[GROUP_COLUMN].dropna().astype(str)),
        'calibration': set(calibration_df[GROUP_COLUMN].dropna().astype(str)),
        'hidden_test': set(test_df[GROUP_COLUMN].dropna().astype(str)),
    }
    group_overlap = {
        'train_calibration': len(group_sets['train'] & group_sets['calibration']),
        'train_hidden_test': len(group_sets['train'] & group_sets['hidden_test']),
        'calibration_hidden_test': len(group_sets['calibration'] & group_sets['hidden_test']),
    }
    validation_report['genetic_group_overlap'] = group_overlap
    if any(group_overlap.values()):
        raise ValueError(f'Genetic-group overlap detected across splits: {group_overlap}')
    print('No genetic-group overlap was detected across the supplied splits.')
else:
    print(
        'WARNING: No genetic-group column was found. The notebook can confirm only that '
        'no exact genome_id overlap exists; it cannot prove sequence-homology isolation.'
    )

# Exact feature-vector overlap is reported as a diagnostic, not automatically called leakage.
def feature_hashes(df):
    return set(pd.util.hash_pandas_object(df[FEATURE_COLUMNS], index=False).astype('uint64').tolist())

feature_hash_sets = {
    'train': feature_hashes(train_df),
    'calibration': feature_hashes(calibration_df),
    'hidden_test': feature_hashes(test_df),
}
validation_report['exact_feature_vector_overlap'] = {
    'train_calibration': len(feature_hash_sets['train'] & feature_hash_sets['calibration']),
    'train_hidden_test': len(feature_hash_sets['train'] & feature_hash_sets['hidden_test']),
    'calibration_hidden_test': len(feature_hash_sets['calibration'] & feature_hash_sets['hidden_test']),
}

print(json.dumps(validation_report, indent=2))

No genetic-group overlap was detected across the supplied splits.


{
  "exact_genome_id_overlap": {
    "train_calibration": 0,
    "train_hidden_test": 0,
    "calibration_hidden_test": 0
  },
  "genome_antibiotic_pair_overlap": {
    "train_calibration": 0,
    "train_hidden_test": 0,
    "calibration_hidden_test": 0
  },
  "genetic_group_column": "cluster_id",
  "genetic_group_overlap": {
    "train_calibration": 0,
    "train_hidden_test": 0,
    "calibration_hidden_test": 0
  },
  "exact_feature_vector_overlap": {
    "train_calibration": 18,
    "train_hidden_test": 20,
    "calibration_hidden_test": 13
  }
}


## Modeling and selective-decision helpers

A base logistic-regression model is fitted on training data. A sigmoid calibrator is then fitted on the calibration subset. The calibrated probabilities are used to choose antibiotic-specific `likely_to_work`, `likely_to_fail`, and `no_call` thresholds.

In [6]:
# 6. Metric and decision helpers
DECISION_WORK = 'likely_to_work'
DECISION_FAIL = 'likely_to_fail'
DECISION_NO_CALL = 'no_call'


def safe_divide(numerator, denominator):
    return float(numerator / denominator) if denominator else np.nan


def make_decisions(probabilities, susceptible_threshold, resistant_threshold):
    probabilities = np.asarray(probabilities, dtype=float)
    decisions = np.full(probabilities.shape, DECISION_NO_CALL, dtype=object)
    decisions[probabilities <= susceptible_threshold] = DECISION_WORK
    decisions[probabilities >= resistant_threshold] = DECISION_FAIL
    return decisions


def binary_metrics(y_true, probabilities):
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    predictions = (probabilities >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    has_both_classes = np.unique(y_true).size == 2

    resistant_recall = safe_divide(tp, tp + fn)
    susceptible_recall = safe_divide(tn, tn + fp)
    balanced_accuracy = (
        float((resistant_recall + susceptible_recall) / 2)
        if has_both_classes else np.nan
    )

    return {
        'binary_prediction': predictions,
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
        'balanced_accuracy': balanced_accuracy,
        'resistant_recall': resistant_recall,
        'susceptible_recall': susceptible_recall,
        'resistant_precision': safe_divide(tp, tp + fp),
        'f1_resistant': (
            float(f1_score(y_true, predictions, pos_label=1, zero_division=0))
            if has_both_classes else np.nan
        ),
        'auroc': float(roc_auc_score(y_true, probabilities)) if has_both_classes else np.nan,
        'pr_auc': float(average_precision_score(y_true, probabilities)) if has_both_classes else np.nan,
        'brier_score': float(brier_score_loss(y_true, probabilities)),
        'full_evaluation_possible': bool(has_both_classes),
    }


def selective_metrics(y_true, decisions):
    y_true = np.asarray(y_true, dtype=int)
    decisions = np.asarray(decisions, dtype=object)
    called_mask = decisions != DECISION_NO_CALL
    fail_mask = decisions == DECISION_FAIL
    work_mask = decisions == DECISION_WORK

    called_predictions = np.full(len(decisions), -1, dtype=int)
    called_predictions[fail_mask] = 1
    called_predictions[work_mask] = 0

    n_total = len(y_true)
    n_called = int(called_mask.sum())
    y_called = y_true[called_mask]
    pred_called = called_predictions[called_mask]

    called_accuracy = (
        float(np.mean(y_called == pred_called)) if n_called else np.nan
    )
    called_balanced_accuracy = (
        float(balanced_accuracy_score(y_called, pred_called))
        if n_called and np.unique(y_called).size == 2 else np.nan
    )

    resistant_total = int((y_true == 1).sum())
    susceptible_total = int((y_true == 0).sum())
    correct_resistant_calls = int(((y_true == 1) & fail_mask).sum())
    correct_susceptible_calls = int(((y_true == 0) & work_mask).sum())

    return {
        'likely_to_work_count': int(work_mask.sum()),
        'likely_to_fail_count': int(fail_mask.sum()),
        'no_call_count': int((~called_mask).sum()),
        'coverage': safe_divide(n_called, n_total),
        'no_call_rate': safe_divide((~called_mask).sum(), n_total),
        'called_accuracy': called_accuracy,
        'called_balanced_accuracy': called_balanced_accuracy,
        'resistant_call_recall': safe_divide(correct_resistant_calls, resistant_total),
        'susceptible_call_recall': safe_divide(correct_susceptible_calls, susceptible_total),
        'precision_likely_to_fail': safe_divide(((y_true == 1) & fail_mask).sum(), fail_mask.sum()),
        'precision_likely_to_work': safe_divide(((y_true == 0) & work_mask).sum(), work_mask.sum()),
        'n_called': n_called,
        'n_called_true_resistant': int(((y_true == 1) & called_mask).sum()),
        'n_called_true_susceptible': int(((y_true == 0) & called_mask).sum()),
    }

In [7]:
# 7. Transparent no-call threshold search
SUSCEPTIBLE_THRESHOLDS = np.round(np.arange(0.05, 0.50, 0.05), 2)
RESISTANT_THRESHOLDS = np.round(np.arange(0.55, 1.00, 0.05), 2)


def select_thresholds(y_true, probabilities):
    rows = []
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)

    for low in SUSCEPTIBLE_THRESHOLDS:
        for high in RESISTANT_THRESHOLDS:
            if low >= high:
                continue
            decisions = make_decisions(probabilities, low, high)
            metrics = selective_metrics(y_true, decisions)
            both_true_classes_called = (
                metrics['n_called_true_resistant'] > 0
                and metrics['n_called_true_susceptible'] > 0
            )
            valid = (
                metrics['n_called'] >= MIN_CALLED_SAMPLES
                and metrics['coverage'] >= MIN_CALIBRATION_COVERAGE
                and not np.isnan(metrics['called_accuracy'])
                and metrics['called_accuracy'] >= TARGET_CALLED_ACCURACY
                and both_true_classes_called
            )
            rows.append({
                'susceptible_threshold': float(low),
                'resistant_threshold': float(high),
                **metrics,
                'both_true_classes_called': both_true_classes_called,
                'meets_all_requirements': bool(valid),
            })

    search_df = pd.DataFrame(rows)
    valid_df = search_df[search_df['meets_all_requirements']].copy()

    if not valid_df.empty:
        selected = valid_df.sort_values(
            ['coverage', 'called_balanced_accuracy', 'resistant_call_recall', 'called_accuracy'],
            ascending=[False, False, False, False],
            na_position='last',
        ).iloc[0]
        requirement_met = True
        failure_reason = ''
    else:
        # Best measured compromise, never an arbitrary threshold fallback.
        eligible = search_df[search_df['n_called'] >= MIN_CALLED_SAMPLES].copy()
        if eligible.empty:
            eligible = search_df[search_df['n_called'] > 0].copy()
        if eligible.empty:
            raise RuntimeError('No threshold pair produced any called predictions.')

        eligible['_called_balanced_rank'] = eligible['called_balanced_accuracy'].fillna(-1)
        eligible['_resistant_recall_rank'] = eligible['resistant_call_recall'].fillna(-1)
        eligible['_called_accuracy_rank'] = eligible['called_accuracy'].fillna(-1)
        selected = eligible.sort_values(
            [
                '_called_balanced_rank', '_resistant_recall_rank',
                '_called_accuracy_rank', 'coverage'
            ],
            ascending=[False, False, False, False],
        ).iloc[0]
        requirement_met = False

        failures = []
        if selected['called_accuracy'] < TARGET_CALLED_ACCURACY:
            failures.append('called accuracy below target')
        if selected['coverage'] < MIN_CALIBRATION_COVERAGE:
            failures.append('coverage below minimum')
        if selected['n_called'] < MIN_CALLED_SAMPLES:
            failures.append('too few called samples')
        if not selected['both_true_classes_called']:
            failures.append('both true classes not represented among calls')
        failure_reason = '; '.join(failures) or 'no threshold pair met all requirements'

    selected_dict = {
        key: selected[key]
        for key in search_df.columns
        if key not in {'meets_all_requirements'}
    }
    selected_dict['threshold_requirement_met'] = bool(requirement_met)
    selected_dict['threshold_failure_reason'] = failure_reason
    return selected_dict, search_df

In [8]:
# 8. Calibration-table and bootstrap helpers

def make_calibration_table(y_true, probabilities, max_bins=10):
    frame = pd.DataFrame({
        'is_resistant': np.asarray(y_true, dtype=int),
        'probability_resistant': np.asarray(probabilities, dtype=float),
    })
    unique_probabilities = frame['probability_resistant'].nunique()
    requested_bins = min(max_bins, len(frame), max(1, unique_probabilities))

    if requested_bins <= 1:
        frame['bin'] = 'single_bin'
    else:
        try:
            frame['bin'] = pd.qcut(
                frame['probability_resistant'],
                q=requested_bins,
                duplicates='drop',
            )
        except ValueError:
            frame['bin'] = 'single_bin'

    table = (
        frame.groupby('bin', observed=True)
             .agg(
                 mean_predicted_probability=('probability_resistant', 'mean'),
                 observed_resistant_fraction=('is_resistant', 'mean'),
                 bin_sample_count=('is_resistant', 'size'),
             )
             .reset_index(drop=True)
    )
    table.insert(0, 'bin_index', np.arange(len(table)))
    return table


def _bootstrap_indices(n, rng, groups=None):
    if groups is None:
        return rng.integers(0, n, size=n)
    groups = np.asarray(groups)
    unique_groups = np.unique(groups)
    sampled_groups = rng.choice(unique_groups, size=len(unique_groups), replace=True)
    positions = [np.flatnonzero(groups == group) for group in sampled_groups]
    return np.concatenate(positions) if positions else np.array([], dtype=int)


def bootstrap_confidence_intervals(
    y_true,
    probabilities,
    susceptible_threshold,
    resistant_threshold,
    groups=None,
    n_bootstraps=N_BOOTSTRAPS,
    random_state=RANDOM_STATE,
):
    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    groups_array = None if groups is None else np.asarray(groups)
    rng = np.random.default_rng(random_state)

    metric_values = {
        'balanced_accuracy': [],
        'resistant_recall': [],
        'susceptible_recall': [],
        'f1_resistant': [],
        'auroc': [],
        'pr_auc': [],
        'brier_score': [],
        'coverage': [],
        'called_accuracy': [],
    }

    for _ in range(n_bootstraps):
        indices = _bootstrap_indices(len(y_true), rng, groups_array)
        if indices.size == 0:
            continue
        y_sample = y_true[indices]
        p_sample = probabilities[indices]
        standard = binary_metrics(y_sample, p_sample)
        decisions = make_decisions(p_sample, susceptible_threshold, resistant_threshold)
        selective = selective_metrics(y_sample, decisions)

        values = {
            'balanced_accuracy': standard['balanced_accuracy'],
            'resistant_recall': standard['resistant_recall'],
            'susceptible_recall': standard['susceptible_recall'],
            'f1_resistant': standard['f1_resistant'],
            'auroc': standard['auroc'],
            'pr_auc': standard['pr_auc'],
            'brier_score': standard['brier_score'],
            'coverage': selective['coverage'],
            'called_accuracy': selective['called_accuracy'],
        }
        for name, value in values.items():
            if value is not None and np.isfinite(value):
                metric_values[name].append(float(value))

    intervals = {}
    for name, values in metric_values.items():
        if len(values) >= max(20, int(0.1 * n_bootstraps)):
            intervals[f'{name}_ci_lower'] = float(np.percentile(values, 2.5))
            intervals[f'{name}_ci_upper'] = float(np.percentile(values, 97.5))
        else:
            intervals[f'{name}_ci_lower'] = np.nan
            intervals[f'{name}_ci_upper'] = np.nan
    intervals['bootstrap_method'] = 'group-level' if groups is not None else 'sample-level'
    intervals['bootstrap_iterations'] = int(n_bootstraps)
    return intervals

## Train and calibrate one model per antibiotic

Antibiotics without both classes in training or calibration are recorded as unsupported. Constant features are removed separately for each antibiotic, and the exact feature order is saved with the model.

In [9]:
# 9. Train and calibrate antibiotic-specific models
all_antibiotics = sorted(
    set(train_df['antibiotic'])
    | set(calibration_df['antibiotic'])
    | set(test_df['antibiotic'])
)

base_models = {}
calibrated_models = {}
feature_lists = {}
thresholds = {}
threshold_search_frames = {}
status_records = []


def class_counts(df, antibiotic):
    subset = df[df['antibiotic'] == antibiotic]
    counts = subset['is_resistant'].value_counts().to_dict()
    return len(subset), int(counts.get(0, 0)), int(counts.get(1, 0))


for antibiotic in all_antibiotics:
    train_subset = train_df[train_df['antibiotic'] == antibiotic].copy()
    calibration_subset = calibration_df[calibration_df['antibiotic'] == antibiotic].copy()
    test_subset = test_df[test_df['antibiotic'] == antibiotic].copy()

    n_train, train_susceptible, train_resistant = class_counts(train_df, antibiotic)
    n_calibration, calibration_susceptible, calibration_resistant = class_counts(calibration_df, antibiotic)
    n_test, test_susceptible, test_resistant = class_counts(test_df, antibiotic)

    status = {
        'antibiotic': antibiotic,
        'n_train': n_train,
        'n_calibration': n_calibration,
        'n_test': n_test,
        'train_susceptible_count': train_susceptible,
        'train_resistant_count': train_resistant,
        'calibration_susceptible_count': calibration_susceptible,
        'calibration_resistant_count': calibration_resistant,
        'test_susceptible_count': test_susceptible,
        'test_resistant_count': test_resistant,
        'base_model_trained': False,
        'calibrated_model_available': False,
        'unsupported_reason': '',
    }

    reasons = []
    if train_susceptible < MIN_CLASS_COUNT_TRAIN or train_resistant < MIN_CLASS_COUNT_TRAIN:
        reasons.append('insufficient class diversity in training')
    if calibration_susceptible < MIN_CLASS_COUNT_CALIBRATION or calibration_resistant < MIN_CLASS_COUNT_CALIBRATION:
        reasons.append('insufficient class diversity in calibration')
    if n_test == 0:
        reasons.append('no hidden-test samples')

    if reasons:
        status['unsupported_reason'] = '; '.join(reasons)
        status_records.append(status)
        continue

    # Remove features constant within this antibiotic's training subset.
    variable_features = [
        col for col in FEATURE_COLUMNS
        if train_subset[col].nunique(dropna=False) > 1
    ]
    if not variable_features:
        status['unsupported_reason'] = 'all features constant in training subset'
        status_records.append(status)
        continue

    X_train = train_subset[variable_features]
    y_train = train_subset['is_resistant'].to_numpy()
    X_calibration = calibration_subset[variable_features]
    y_calibration = calibration_subset['is_resistant'].to_numpy()

    base_model = LogisticRegression(
        penalty='l2',
        class_weight='balanced',
        solver='liblinear',
        max_iter=5000,
        random_state=RANDOM_STATE,
    )
    base_model.fit(X_train, y_train)
    status['base_model_trained'] = True

    if HAS_FROZEN_ESTIMATOR:
        calibrator = CalibratedClassifierCV(
            FrozenEstimator(base_model),
            method='sigmoid',
        )
    else:
        warnings.warn(
            'FrozenEstimator is unavailable; using deprecated cv="prefit" compatibility mode.'
        )
        calibrator = CalibratedClassifierCV(
            base_model,
            method='sigmoid',
            cv='prefit',
        )

    calibrator.fit(X_calibration, y_calibration)
    calibration_probabilities = calibrator.predict_proba(X_calibration)[:, 1]
    selected_thresholds, search_df = select_thresholds(
        y_calibration,
        calibration_probabilities,
    )

    base_models[antibiotic] = base_model
    calibrated_models[antibiotic] = calibrator
    feature_lists[antibiotic] = variable_features
    thresholds[antibiotic] = selected_thresholds
    threshold_search_frames[antibiotic] = search_df
    status['calibrated_model_available'] = True
    status['n_features'] = len(variable_features)
    status['unsupported_reason'] = ''
    status_records.append(status)

status_df = pd.DataFrame(status_records).sort_values('antibiotic').reset_index(drop=True)
print('Base models trained:', int(status_df['base_model_trained'].sum()))
print('Calibrated models available:', int(status_df['calibrated_model_available'].sum()))
print('Unsupported antibiotics:', int((~status_df['calibrated_model_available']).sum()))
display(status_df)

/var/folders/r1/qcp7_6gx64b7h09ztlk5y3pm0000gp/T/ipykernel_39438/2435680231.py:91: UserWarning: FrozenEstimator is unavailable; using deprecated cv="prefit" compatibility mode.
  warnings.warn(
/var/folders/r1/qcp7_6gx64b7h09ztlk5y3pm0000gp/T/ipykernel_39438/2435680231.py:91: UserWarning: FrozenEstimator is unavailable; using deprecated cv="prefit" compatibility mode.
  warnings.warn(


/var/folders/r1/qcp7_6gx64b7h09ztlk5y3pm0000gp/T/ipykernel_39438/2435680231.py:91: UserWarning: FrozenEstimator is unavailable; using deprecated cv="prefit" compatibility mode.
  warnings.warn(
/var/folders/r1/qcp7_6gx64b7h09ztlk5y3pm0000gp/T/ipykernel_39438/2435680231.py:91: UserWarning: FrozenEstimator is unavailable; using deprecated cv="prefit" compatibility mode.
  warnings.warn(


/var/folders/r1/qcp7_6gx64b7h09ztlk5y3pm0000gp/T/ipykernel_39438/2435680231.py:91: UserWarning: FrozenEstimator is unavailable; using deprecated cv="prefit" compatibility mode.
  warnings.warn(
/var/folders/r1/qcp7_6gx64b7h09ztlk5y3pm0000gp/T/ipykernel_39438/2435680231.py:91: UserWarning: FrozenEstimator is unavailable; using deprecated cv="prefit" compatibility mode.
  warnings.warn(


/var/folders/r1/qcp7_6gx64b7h09ztlk5y3pm0000gp/T/ipykernel_39438/2435680231.py:91: UserWarning: FrozenEstimator is unavailable; using deprecated cv="prefit" compatibility mode.
  warnings.warn(
/var/folders/r1/qcp7_6gx64b7h09ztlk5y3pm0000gp/T/ipykernel_39438/2435680231.py:91: UserWarning: FrozenEstimator is unavailable; using deprecated cv="prefit" compatibility mode.
  warnings.warn(


/var/folders/r1/qcp7_6gx64b7h09ztlk5y3pm0000gp/T/ipykernel_39438/2435680231.py:91: UserWarning: FrozenEstimator is unavailable; using deprecated cv="prefit" compatibility mode.
  warnings.warn(
/var/folders/r1/qcp7_6gx64b7h09ztlk5y3pm0000gp/T/ipykernel_39438/2435680231.py:91: UserWarning: FrozenEstimator is unavailable; using deprecated cv="prefit" compatibility mode.
  warnings.warn(


...

/var/folders/r1/qcp7_6gx64b7h09ztlk5y3pm0000gp/T/ipykernel_39438/2435680231.py:91: UserWarning: FrozenEstimator is unavailable; using deprecated cv="prefit" compatibility mode.
  warnings.warn(
/var/folders/r1/qcp7_6gx64b7h09ztlk5y3pm0000gp/T/ipykernel_39438/2435680231.py:91: UserWarning: FrozenEstimator is unavailable; using deprecated cv="prefit" compatibility mode.
  warnings.warn(


/var/folders/r1/qcp7_6gx64b7h09ztlk5y3pm0000gp/T/ipykernel_39438/2435680231.py:91: UserWarning: FrozenEstimator is unavailable; using deprecated cv="prefit" compatibility mode.
  warnings.warn(
/var/folders/r1/qcp7_6gx64b7h09ztlk5y3pm0000gp/T/ipykernel_39438/2435680231.py:91: UserWarning: FrozenEstimator is unavailable; using deprecated cv="prefit" compatibility mode.
  warnings.warn(


/var/folders/r1/qcp7_6gx64b7h09ztlk5y3pm0000gp/T/ipykernel_39438/2435680231.py:91: UserWarning: FrozenEstimator is unavailable; using deprecated cv="prefit" compatibility mode.
  warnings.warn(
/var/folders/r1/qcp7_6gx64b7h09ztlk5y3pm0000gp/T/ipykernel_39438/2435680231.py:91: UserWarning: FrozenEstimator is unavailable; using deprecated cv="prefit" compatibility mode.
  warnings.warn(


Base models trained: 23
Calibrated models available: 23
Unsupported antibiotics: 40


,antibiotic,n_train,n_calibration,n_test,train_susceptible_count,train_resistant_count,calibration_susceptible_count,calibration_resistant_count,test_susceptible_count,test_resistant_count,base_model_trained,calibrated_model_available,unsupported_reason,n_features
0,amikacin,186,39,46,183,3,38,1,46,0,False,False,insufficient class diversity in calibration,NaN
1,amoxicillin,44,23,17,15,29,12,11,8,9,True,True,,32.0
2,amoxicillin/clavulanic acid,186,70,70,131,55,51,19,45,25,True,True,,92.0
3,ampicillin,557,232,210,259,298,104,128,91,119,True,True,,125.0
4,ampicillin/sulbactam,33,4,5,19,14,1,3,1,4,False,False,insufficient class diversity in calibration,NaN
5,azithromycin,77,3,7,72,5,3,0,7,0,False,False,insufficient class diversity in calibration,NaN
6,aztreonam,104,21,22,90,14,17,4,20,2,True,True,,66.0
7,beta-lactam,0,1,0,0,0,0,1,0,0,False,False,insufficient class diversity in training; insu...,NaN
8,carbenicillin,0,1,1,0,0,0,1,0,1,False,False,insufficient class diversity in training; insu...,NaN
9,cefalotin,18,3,8,15,3,2,1,7,1,False,False,insufficient class diversity in calibration,NaN


In [10]:
# 10. Final hidden-test evaluation — do not tune anything after this point
summary_rows = []
prediction_rows = []
calibration_tables = {}

for antibiotic, model in calibrated_models.items():
    test_subset = test_df[test_df['antibiotic'] == antibiotic].copy()
    features = feature_lists[antibiotic]
    threshold_info = thresholds[antibiotic]
    low = float(threshold_info['susceptible_threshold'])
    high = float(threshold_info['resistant_threshold'])

    X_test = test_subset[features]
    y_test = test_subset['is_resistant'].to_numpy()
    probabilities = model.predict_proba(X_test)[:, 1]

    standard = binary_metrics(y_test, probabilities)
    decisions = make_decisions(probabilities, low, high)
    selective = selective_metrics(y_test, decisions)

    group_values = test_subset[GROUP_COLUMN].to_numpy() if GROUP_COLUMN else None
    antibiotic_seed = RANDOM_STATE + (zlib.crc32(str(antibiotic).encode('utf-8')) % 100_000)
    intervals = bootstrap_confidence_intervals(
        y_test,
        probabilities,
        low,
        high,
        groups=group_values,
        random_state=antibiotic_seed,
    )

    n_resistant = int((y_test == 1).sum())
    n_susceptible = int((y_test == 0).sum())
    full_evaluation = bool(standard['full_evaluation_possible'])
    evaluation_warning = '' if full_evaluation else (
        'Hidden-test subset contains only one class; metrics requiring both classes are undefined.'
    )

    threshold_requirement_met = bool(threshold_info['threshold_requirement_met'])

    # Post-evaluation descriptive flag only. Do not use it to retune the pipeline.
    deployment_candidate = bool(
        full_evaluation
        and len(test_subset) >= MIN_TEST_SAMPLES_FOR_DEPLOYMENT_FLAG
        and threshold_requirement_met
        and np.isfinite(standard['balanced_accuracy'])
        and standard['balanced_accuracy'] >= MIN_BALANCED_ACCURACY
        and np.isfinite(standard['resistant_recall'])
        and standard['resistant_recall'] >= MIN_RESISTANT_RECALL
        and np.isfinite(selective['coverage'])
        and selective['coverage'] >= MIN_DEPLOYMENT_COVERAGE
        and np.isfinite(selective['called_accuracy'])
        and selective['called_accuracy'] >= MIN_DEPLOYMENT_CALLED_ACCURACY
    )

    row = {
        'antibiotic': antibiotic,
        'model_status': 'calibrated',
        'unsupported_reason': '',
        'n_train': int((train_df['antibiotic'] == antibiotic).sum()),
        'n_calibration': int((calibration_df['antibiotic'] == antibiotic).sum()),
        'n_test': int(len(test_subset)),
        'n_features': int(len(features)),
        'test_resistant_count': n_resistant,
        'test_susceptible_count': n_susceptible,
        'test_resistant_prevalence': safe_divide(n_resistant, len(test_subset)),
        'susceptible_threshold': low,
        'resistant_threshold': high,
        'threshold_requirement_met': threshold_requirement_met,
        'threshold_failure_reason': threshold_info['threshold_failure_reason'],
        'full_evaluation_possible': full_evaluation,
        'evaluation_warning': evaluation_warning,
        'deployment_candidate': deployment_candidate,
        **{k: v for k, v in standard.items() if k != 'binary_prediction'},
        **selective,
        **intervals,
    }
    summary_rows.append(row)

    binary_predictions = standard['binary_prediction']
    for position, (_, source_row) in enumerate(test_subset.iterrows()):
        decision = decisions[position]
        true_label = int(source_row['is_resistant'])
        is_no_call = decision == DECISION_NO_CALL
        if is_no_call:
            correct_if_called = np.nan
        else:
            predicted_label = 1 if decision == DECISION_FAIL else 0
            correct_if_called = bool(predicted_label == true_label)

        prediction_rows.append({
            'genome_id': str(source_row['genome_id']),
            'antibiotic': antibiotic,
            'is_resistant': true_label,
            'probability_resistant': float(probabilities[position]),
            'binary_prediction_at_0_5': int(binary_predictions[position]),
            'final_decision': decision,
            'is_no_call': is_no_call,
            'is_correct_if_called': correct_if_called,
            'susceptible_threshold': low,
            'resistant_threshold': high,
            'threshold_requirement_met': threshold_requirement_met,
            'model_status': 'calibrated',
            'deployment_candidate': deployment_candidate,
            **({GROUP_COLUMN: source_row[GROUP_COLUMN]} if GROUP_COLUMN else {}),
        })

    calibration_tables[antibiotic] = make_calibration_table(y_test, probabilities)

summary_df = pd.DataFrame(summary_rows).sort_values('antibiotic').reset_index(drop=True)
predictions_df = pd.DataFrame(prediction_rows)

# Append unsupported antibiotics to create one complete summary row per antibiotic.
unsupported_status = status_df[~status_df['calibrated_model_available']].copy()
if not unsupported_status.empty:
    unsupported_rows = []
    for _, item in unsupported_status.iterrows():
        unsupported_rows.append({
            'antibiotic': item['antibiotic'],
            'model_status': 'unsupported',
            'unsupported_reason': item['unsupported_reason'],
            'n_train': item['n_train'],
            'n_calibration': item['n_calibration'],
            'n_test': item['n_test'],
            'deployment_candidate': False,
            'full_evaluation_possible': False,
        })
    summary_df = pd.concat([summary_df, pd.DataFrame(unsupported_rows)], ignore_index=True)
    summary_df = summary_df.sort_values('antibiotic').reset_index(drop=True)

print('Evaluated calibrated antibiotics:', len(calibrated_models))
print('Fully evaluable antibiotics:', int(summary_df['full_evaluation_possible'].fillna(False).sum()))
print('Deployment candidates:', int(summary_df['deployment_candidate'].fillna(False).sum()))
display(summary_df)

Evaluated calibrated antibiotics: 23
Fully evaluable antibiotics: 23
Deployment candidates: 0


,antibiotic,model_status,unsupported_reason,n_train,n_calibration,n_test,n_features,test_resistant_count,test_susceptible_count,test_resistant_prevalence,susceptible_threshold,resistant_threshold,threshold_requirement_met,threshold_failure_reason,full_evaluation_possible,evaluation_warning,deployment_candidate,tn,fp,fn,tp,balanced_accuracy,resistant_recall,susceptible_recall,resistant_precision,f1_resistant,auroc,pr_auc,brier_score,likely_to_work_count,likely_to_fail_count,no_call_count,coverage,no_call_rate,called_accuracy,called_balanced_accuracy,resistant_call_recall,susceptible_call_recall,precision_likely_to_fail,precision_likely_to_work,n_called,n_called_true_resistant,n_called_true_susceptible,balanced_accuracy_ci_lower,balanced_accuracy_ci_upper,resistant_recall_ci_lower,resistant_recall_ci_upper,susceptible_recall_ci_lower,susceptible_recall_ci_upper,f1_resistant_ci_lower,f1_resistant_ci_upper,auroc_ci_lower,auroc_ci_upper,pr_auc_ci_lower,pr_auc_ci_upper,brier_score_ci_lower,brier_score_ci_upper,coverage_ci_lower,coverage_ci_upper,called_accuracy_ci_lower,called_accuracy_ci_upper,bootstrap_method,bootstrap_iterations
0,amikacin,unsupported,insufficient class diversity in calibration,186,39,46,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,amoxicillin,calibrated,,44,23,17,32.0,9.0,8.0,0.529412,0.35,0.55,False,called accuracy below target,True,,False,8.0,0.0,3.0,6.0,0.833333,0.666667,1.000000,1.000000,0.800000,0.909722,0.906746,0.152783,3.0,6.0,8.0,0.529412,0.470588,1.000000,1.000000,0.666667,0.375000,1.000000,1.000000,9.0,6.0,3.0,0.646250,1.000000,0.292500,1.000000,1.000000,1.000000,0.452564,1.000000,0.764489,1.000000,0.724062,1.000000,0.097157,0.222007,0.294118,0.764706,1.000000,1.000000,group-level,500.0
2,amoxicillin/clavulanic acid,calibrated,,186,70,70,92.0,25.0,45.0,0.357143,0.15,0.55,False,called accuracy below target; coverage below m...,True,,False,44.0,1.0,19.0,6.0,0.608889,0.240000,0.977778,0.857143,0.375000,0.509778,0.506939,0.234211,15.0,5.0,50.0,0.285714,0.714286,0.650000,0.650000,0.160000,0.200000,0.800000,0.600000,20.0,10.0,10.0,0.528306,0.690086,0.083333,0.408291,0.927657,1.000000,0.142857,0.561270,0.360882,0.663770,0.340112,0.668218,0.176100,0.307427,0.185714,0.385714,0.421053,0.866667,group-level,500.0
3,ampicillin,calibrated,,557,232,210,125.0,119.0,91.0,0.566667,0.25,0.85,False,called accuracy below target; coverage below m...,True,,False,37.0,54.0,23.0,96.0,0.606658,0.806723,0.406593,0.640000,0.713755,0.683073,0.749447,0.219090,2.0,11.0,197.0,0.061905,0.938095,1.000000,1.000000,0.092437,0.021978,1.000000,1.000000,13.0,11.0,2.0,0.542885,0.673686,0.728775,0.883541,0.312800,0.515999,0.648522,0.778247,0.610410,0.754808,0.678958,0.811035,0.202482,0.235965,0.033019,0.095694,1.000000,1.000000,group-level,500.0
4,ampicillin/sulbactam,unsupported,insufficient class diversity in calibration,33,4,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,azithromycin,unsupported,insufficient class diversity in calibration,77,3,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,aztreonam,calibrated,,104,21,22,66.0,2.0,20.0,0.090909,0.25,0.55,False,called accuracy below target,True,,False,20.0,0.0,2.0,0.0,0.500000,0.000000,1.000000,NaN,0.000000,0.600000,0.229167,0.094736,22.0,0.0,0.0,1.000000,0.000000,0.909091,0.500000,0.000000,1.000000,NaN,0.909091,22.0,2.0,20.0,0.500000,0.500000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.151875,0.952381,0.055556,0.818236,0.

In [11]:
# 11. Save reliability, confusion-matrix, ROC, PR, probability, and threshold-tradeoff plots
plot_warnings = []

for antibiotic, model in calibrated_models.items():
    test_subset = test_df[test_df['antibiotic'] == antibiotic]
    features = feature_lists[antibiotic]
    y_test = test_subset['is_resistant'].to_numpy()
    probabilities = model.predict_proba(test_subset[features])[:, 1]
    predictions = (probabilities >= 0.5).astype(int)
    table = calibration_tables[antibiotic]
    brier = brier_score_loss(y_test, probabilities)

    # Reliability plot
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], linestyle='--', label='Perfect calibration')
    ax.plot(
        table['mean_predicted_probability'],
        table['observed_resistant_fraction'],
        marker='o',
        label='Calibrated model',
    )
    for _, bin_row in table.iterrows():
        ax.annotate(
            f"n={int(bin_row['bin_sample_count'])}",
            (bin_row['mean_predicted_probability'], bin_row['observed_resistant_fraction']),
            textcoords='offset points',
            xytext=(4, 4),
            fontsize=8,
        )
    ax.set_xlabel('Mean predicted probability of resistance')
    ax.set_ylabel('Observed resistant fraction')
    ax.set_title(
        f'Reliability: {antibiotic}\n'
        f'Brier={brier:.3f}, n={len(test_subset)}, '
        f'R={int((y_test == 1).sum())}, S={int((y_test == 0).sum())}'
    )
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'plots' / 'reliability' / f'reliability_{antibiotic.replace("/", "_")}.png', dpi=160)
    plt.close(fig)

    if len(test_subset) < 30 or len(table) < 4:
        plot_warnings.append({
            'antibiotic': antibiotic,
            'warning': 'Reliability plot is unstable because the test subset or occupied-bin count is small.',
            'n_test': len(test_subset),
            'occupied_bins': len(table),
        })

    # Confusion matrix
    cm = confusion_matrix(y_test, predictions, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(5, 5))
    image = ax.imshow(cm)
    for (i, j), value in np.ndenumerate(cm):
        ax.text(j, i, str(value), ha='center', va='center')
    ax.set_xticks([0, 1], labels=['Predicted S', 'Predicted R'])
    ax.set_yticks([0, 1], labels=['True S', 'True R'])
    ax.set_title(f'Confusion matrix: {antibiotic}')
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'plots' / 'confusion_matrices' / f'confusion_{antibiotic.replace("/", "_")}.png', dpi=160)
    plt.close(fig)

    # ROC and precision-recall curves only when both classes are present.
    if np.unique(y_test).size == 2:
        fpr, tpr, _ = roc_curve(y_test, probabilities)
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.plot(fpr, tpr, label=f'AUROC={roc_auc_score(y_test, probabilities):.3f}')
        ax.plot([0, 1], [0, 1], linestyle='--')
        ax.set_xlabel('False positive rate')
        ax.set_ylabel('True positive rate')
        ax.set_title(f'ROC: {antibiotic}')
        ax.legend()
        fig.tight_layout()
        fig.savefig(OUTPUT_DIR / 'plots' / 'roc' / f'roc_{antibiotic.replace("/", "_")}.png', dpi=160)
        plt.close(fig)

        precision, recall, _ = precision_recall_curve(y_test, probabilities)
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.plot(recall, precision, label=f'PR-AUC={average_precision_score(y_test, probabilities):.3f}')
        ax.set_xlabel('Recall')
        ax.set_ylabel('Precision')
        ax.set_title(f'Precision–recall: {antibiotic}')
        ax.legend()
        fig.tight_layout()
        fig.savefig(OUTPUT_DIR / 'plots' / 'precision_recall' / f'pr_{antibiotic.replace("/", "_")}.png', dpi=160)
        plt.close(fig)

    # Probability distributions by true class
    fig, ax = plt.subplots(figsize=(6, 5))
    for label, label_name in [(0, 'Susceptible'), (1, 'Resistant')]:
        values = probabilities[y_test == label]
        if len(values):
            ax.hist(values, bins=10, alpha=0.6, label=f'{label_name} (n={len(values)})')
    ax.set_xlabel('Predicted probability of resistance')
    ax.set_ylabel('Count')
    ax.set_title(f'Probability distribution: {antibiotic}')
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'plots' / 'probability_distributions' / f'probability_{antibiotic.replace("/", "_")}.png', dpi=160)
    plt.close(fig)

    # Calibration threshold trade-off plot
    search_df = threshold_search_frames[antibiotic]
    selected_low = thresholds[antibiotic]['susceptible_threshold']
    selected_high = thresholds[antibiotic]['resistant_threshold']
    selected_rows = search_df[
        (search_df['susceptible_threshold'] == selected_low)
        | (search_df['resistant_threshold'] == selected_high)
    ].copy()
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(search_df['coverage'], search_df['called_accuracy'], s=18, alpha=0.6)
    chosen = search_df[
        (search_df['susceptible_threshold'] == selected_low)
        & (search_df['resistant_threshold'] == selected_high)
    ]
    if not chosen.empty:
        ax.scatter(chosen['coverage'], chosen['called_accuracy'], s=90, label='Selected thresholds')
    ax.axhline(TARGET_CALLED_ACCURACY, linestyle='--', label='Target called accuracy')
    ax.axvline(MIN_CALIBRATION_COVERAGE, linestyle=':', label='Minimum coverage')
    ax.set_xlabel('Calibration coverage')
    ax.set_ylabel('Calibration called-case accuracy')
    ax.set_title(f'Threshold trade-off: {antibiotic}')
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'plots' / 'threshold_tradeoffs' / f'thresholds_{antibiotic.replace("/", "_")}.png', dpi=160)
    plt.close(fig)

print('Plots saved. Reliability warnings:', len(plot_warnings))
if plot_warnings:
    display(pd.DataFrame(plot_warnings))

Plots saved. Reliability warnings: 12


,antibiotic,warning,n_test,occupied_bins
0,amoxicillin,Reliability plot is unstable because the test ...,17,7
1,aztreonam,Reliability plot is unstable because the test ...,22,8
2,cefepime,Reliability plot is unstable because the test ...,16,9
3,cefoxitin,Reliability plot is unstable because the test ...,14,4
4,ceftriaxone,Reliability plot is unstable because the test ...,23,8
5,cephalothin,Reliability plot is unstable because the test ...,8,3
6,chloramphenicol,Reliability plot is unstable because the test ...,10,5
7,ertapenem,Reliability plot is unstable because the test ...,16,9
8,imipenem,Reliability plot is unstable because the test ...,27,8
9,tetracycline,Reliability plot is unstable because the test ...,11,6


In [12]:
# 12. Genetic-group performance, where available
per_group_rows = []

if GROUP_COLUMN:
    for antibiotic, model in calibrated_models.items():
        subset = test_df[test_df['antibiotic'] == antibiotic].copy()
        features = feature_lists[antibiotic]
        probabilities_all = model.predict_proba(subset[features])[:, 1]
        subset = subset.assign(probability_resistant=probabilities_all)
        low = thresholds[antibiotic]['susceptible_threshold']
        high = thresholds[antibiotic]['resistant_threshold']

        for group_value, group_df in subset.groupby(GROUP_COLUMN, observed=True):
            y_group = group_df['is_resistant'].to_numpy()
            p_group = group_df['probability_resistant'].to_numpy()
            standard = binary_metrics(y_group, p_group)
            decisions = make_decisions(p_group, low, high)
            selective = selective_metrics(y_group, decisions)
            per_group_rows.append({
                'antibiotic': antibiotic,
                GROUP_COLUMN: group_value,
                'n_samples': len(group_df),
                'small_group_warning': len(group_df) < 10,
                'resistant_prevalence': float(np.mean(y_group)),
                **{k: v for k, v in standard.items() if k != 'binary_prediction'},
                **selective,
            })

    per_group_df = pd.DataFrame(per_group_rows)
    display(per_group_df.head())
else:
    per_group_df = pd.DataFrame()
    warning_text = (
        'Generalization by genetic group could not be evaluated because the supplied CSV files '
        'do not contain a recognized genetic-group identifier. The fixed organizer split was used, '
        'and no exact genome_id overlap was detected, but a cluster mapping is required for '
        'sequence-homology generalization reporting.'
    )
    print('WARNING:', warning_text)
    (OUTPUT_DIR / 'metrics' / 'genetic_group_warning.txt').write_text(warning_text, encoding='utf-8')

,antibiotic,cluster_id,n_samples,small_group_warning,resistant_prevalence,tn,fp,fn,tp,balanced_accuracy,resistant_recall,susceptible_recall,resistant_precision,f1_resistant,auroc,pr_auc,brier_score,full_evaluation_possible,likely_to_work_count,likely_to_fail_count,no_call_count,coverage,no_call_rate,called_accuracy,called_balanced_accuracy,resistant_call_recall,susceptible_call_recall,precision_likely_to_fail,precision_likely_to_work,n_called,n_called_true_resistant,n_called_true_susceptible
0,amoxicillin,669,1,True,0.0,1,0,0,0,NaN,NaN,1.0,NaN,NaN,NaN,NaN,0.110107,False,1,0,0,1.0,0.0,1.0,NaN,NaN,1.0,NaN,1.0,1,0,1
1,amoxicillin,672,1,True,1.0,0,0,1,0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.404969,False,0,0,1,0.0,1.0,NaN,NaN,0.0,NaN,NaN,NaN,0,0,0
2,amoxicillin,678,1,True,0.0,1,0,0,0,NaN,NaN,1.0,NaN,NaN,NaN,NaN,0.132225,False,0,0,1,0.0,1.0,NaN,NaN,NaN,0.0,NaN,NaN,0,0,0
3,amoxicillin,686,1,True,1.0,0,0,1,0,NaN,0.0,NaN,NaN,NaN,NaN,NaN,0.404925,False,0,0,1,0.0,1.0,NaN,NaN,0.0,NaN,NaN,NaN,0,0,0
4,amoxicillin,699,1,True,0.0,1,0,0,0,NaN,NaN,1.0,NaN,NaN,NaN,NaN,0.132225,False,0,0,1,0.0,1.0,NaN,NaN,NaN,0.0,NaN,NaN,0,0,0


In [13]:
# 13. Coefficient tables — statistical association, not biological causation
all_coefficient_rows = []
calibrated_coefficient_rows = []

for antibiotic, base_model in base_models.items():
    features = feature_lists[antibiotic]
    for feature, coefficient in zip(features, base_model.coef_[0]):
        row = {
            'antibiotic': antibiotic,
            'feature': feature,
            'coefficient': float(coefficient),
            'absolute_coefficient': float(abs(coefficient)),
            'direction': (
                'statistically associated with higher resistance probability'
                if coefficient > 0
                else 'statistically associated with lower resistance probability'
                if coefficient < 0
                else 'no fitted association'
            ),
            'causation_warning': 'Coefficient is a statistical association and does not prove biological causation.',
        }
        all_coefficient_rows.append(row)
        if antibiotic in calibrated_models:
            calibrated_coefficient_rows.append(row.copy())

all_coefficients_df = pd.DataFrame(all_coefficient_rows)
calibrated_coefficients_df = pd.DataFrame(calibrated_coefficient_rows)

if not calibrated_coefficients_df.empty:
    top_associations_df = (
        calibrated_coefficients_df
        .sort_values(['antibiotic', 'absolute_coefficient'], ascending=[True, False])
        .groupby('antibiotic', observed=True)
        .head(20)
        .reset_index(drop=True)
    )
else:
    top_associations_df = pd.DataFrame()

display(top_associations_df.head(20))

,antibiotic,feature,coefficient,absolute_coefficient,direction,causation_warning
0,amoxicillin,gene_blaEC,1.082149,1.082149,statistically associated with higher resistanc...,Coefficient is a statistical association and d...
1,amoxicillin,gene_blaEC-5,-1.001391,1.001391,statistically associated with lower resistance...,Coefficient is a statistical association and d...
2,amoxicillin,gene_tet(A),0.929088,0.929088,statistically associated with higher resistanc...,Coefficient is a statistical association and d...
3,amoxicillin,mut_cyaA_S352T,0.564745,0.564745,statistically associated with higher resistanc...,Coefficient is a statistical association and d...
4,amoxicillin,gene_aph(3'')-Ib,0.497347,0.497347,statistically associated with higher resistanc...,Coefficient is a statistical association and d...
5,amoxicillin,gene_aph(6)-Id,0.497347,0.497347,statistically associated with higher resistanc...,Coefficient is a statistical association and d...
6,amoxicillin,gene_sul2,0.497347,0.497347,statistically associated with higher resistanc...,Coefficient is a statistical association and d...
7,amoxicillin,mut_gyrA_S83L,0.496773,0.496773,statistically associated with higher resistanc...,Coefficient is a statistical association and d...
8,amoxicillin,mut_marR_S3N,-0.486459,0.486459,statistically associated with lower resistance...,Coefficient is a statistical association and d...
9,amoxicillin,gene_aadA5,0.401261,0.401261,statistically associated with higher resistanc...,Coefficient is a statistical association and d...


In [14]:
# 14. Save all artifacts and reports

def to_native(value):
    if isinstance(value, dict):
        return {str(k): to_native(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_native(v) for v in value]
    if isinstance(value, np.ndarray):
        return [to_native(v) for v in value.tolist()]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if value is pd.NA:
        return None
    if isinstance(value, float) and (math.isnan(value) or math.isinf(value)):
        return None
    return value

# Model artifacts
joblib.dump(calibrated_models, OUTPUT_DIR / 'models' / 'calibrated_models.joblib')
joblib.dump(base_models, OUTPUT_DIR / 'models' / 'base_models.joblib')
joblib.dump(feature_lists, OUTPUT_DIR / 'features' / 'feature_lists.joblib')
joblib.dump({
    'calibrated_models': calibrated_models,
    'feature_lists': feature_lists,
    'thresholds': thresholds,
}, OUTPUT_DIR / 'models' / 'model_bundle.joblib')

with open(OUTPUT_DIR / 'thresholds' / 'thresholds.json', 'w', encoding='utf-8') as file:
    json.dump(to_native(thresholds), file, indent=2, sort_keys=True)

# Threshold searches and calibration tables
for antibiotic, frame in threshold_search_frames.items():
    safe_name = antibiotic.replace('/', '_')
    frame.to_csv(OUTPUT_DIR / 'threshold_search' / f'{safe_name}.csv', index=False)
for antibiotic, frame in calibration_tables.items():
    safe_name = antibiotic.replace('/', '_')
    frame.to_csv(OUTPUT_DIR / 'calibration_tables' / f'{safe_name}.csv', index=False)

# Main reports
summary_df.to_csv(OUTPUT_DIR / 'metrics' / 'final_summary_report.csv', index=False)
with open(OUTPUT_DIR / 'metrics' / 'final_summary_report.json', 'w', encoding='utf-8') as file:
    json.dump(to_native(summary_df.to_dict(orient='records')), file, indent=2)

status_df.to_csv(OUTPUT_DIR / 'metrics' / 'model_status.csv', index=False)
balance_table.to_csv(OUTPUT_DIR / 'metrics' / 'class_balance_by_split.csv', index=False)
duplicate_report.to_csv(OUTPUT_DIR / 'metrics' / 'duplicate_report.csv', index=False)
predictions_df.to_csv(OUTPUT_DIR / 'predictions' / 'hidden_test_predictions.csv', index=False)

supported_df = status_df[status_df['calibrated_model_available']].copy()
unsupported_df = status_df[~status_df['calibrated_model_available']].copy()
deployment_candidates_df = summary_df[summary_df['deployment_candidate'].fillna(False)].copy()

supported_df.to_csv(OUTPUT_DIR / 'metrics' / 'supported_antibiotics.csv', index=False)
unsupported_df.to_csv(OUTPUT_DIR / 'metrics' / 'unsupported_antibiotics.csv', index=False)
deployment_candidates_df.to_csv(OUTPUT_DIR / 'metrics' / 'deployment_candidates.csv', index=False)

if not per_group_df.empty:
    per_group_df.to_csv(OUTPUT_DIR / 'metrics' / 'per_genetic_group_metrics.csv', index=False)

all_coefficients_df.to_csv(OUTPUT_DIR / 'explanations' / 'all_base_model_coefficients.csv', index=False)
calibrated_coefficients_df.to_csv(OUTPUT_DIR / 'explanations' / 'calibrated_model_coefficients.csv', index=False)
top_associations_df.to_csv(OUTPUT_DIR / 'explanations' / 'top_statistical_associations.csv', index=False)

validation_report_path = OUTPUT_DIR / 'metrics' / 'validation_report.json'
with open(validation_report_path, 'w', encoding='utf-8') as file:
    json.dump(to_native(validation_report), file, indent=2, sort_keys=True)

metadata = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'python_version': platform.python_version(),
    'pandas_version': pd.__version__,
    'numpy_version': np.__version__,
    'scikit_learn_version': sklearn.__version__,
    'joblib_version': joblib.__version__,
    'random_state': RANDOM_STATE,
    'bootstrap_iterations': N_BOOTSTRAPS,
    'bootstrap_method': 'group-level' if GROUP_COLUMN else 'sample-level',
    'base_path': str(BASE_PATH),
    'train_path': str(TRAIN_PATH),
    'calibration_path': str(CALIBRATION_PATH),
    'hidden_test_path': str(TEST_PATH),
    'output_dir': str(OUTPUT_DIR),
    'global_feature_count_before_antibiotic_specific_constant_removal': len(FEATURE_COLUMNS),
    'base_model_count': len(base_models),
    'calibrated_model_count': len(calibrated_models),
    'unsupported_antibiotics': unsupported_df['antibiotic'].tolist(),
    'supported_antibiotics': supported_df['antibiotic'].tolist(),
    'deployment_candidates': deployment_candidates_df['antibiotic'].tolist(),
    'genetic_group_column': GROUP_COLUMN,
    'threshold_policy': {
        'target_called_accuracy': TARGET_CALLED_ACCURACY,
        'minimum_calibration_coverage': MIN_CALIBRATION_COVERAGE,
        'minimum_called_samples': MIN_CALLED_SAMPLES,
    },
    'safety_warning': 'Research prototype. Every prediction must be confirmed with standard laboratory susceptibility testing.',
}
with open(OUTPUT_DIR / 'metadata.json', 'w', encoding='utf-8') as file:
    json.dump(to_native(metadata), file, indent=2, sort_keys=True)

print('All artifacts saved to:', OUTPUT_DIR)
print('Calibrated models:', len(calibrated_models))
print('Unsupported antibiotics:', len(unsupported_df))
print('Fully evaluable antibiotics:', int(summary_df['full_evaluation_possible'].fillna(False).sum()))
print('Deployment candidates:', len(deployment_candidates_df))

All artifacts saved to: model_outputs_final
Calibrated models: 23
Unsupported antibiotics: 40
Fully evaluable antibiotics: 23
Deployment candidates: 0


In [15]:
# 15. Reusable prediction interface for the app
RESEARCH_WARNING = (
    'Research prototype. Every prediction must be confirmed with standard '
    'laboratory susceptibility testing.'
)


def load_prediction_artifacts(output_dir=OUTPUT_DIR):
    output_dir = Path(output_dir)
    models = joblib.load(output_dir / 'models' / 'calibrated_models.joblib')
    features = joblib.load(output_dir / 'features' / 'feature_lists.joblib')
    with open(output_dir / 'thresholds' / 'thresholds.json', encoding='utf-8') as file:
        loaded_thresholds = json.load(file)
    summary = pd.read_csv(output_dir / 'metrics' / 'final_summary_report.csv')
    return models, features, loaded_thresholds, summary


def predict_resistance(
    feature_row,
    antibiotic,
    output_dir=OUTPUT_DIR,
    target_gate_passed=None,
):
    models, features_by_antibiotic, threshold_data, summary = load_prediction_artifacts(output_dir)

    exact_names = {str(name).lower(): name for name in models}
    key = str(antibiotic).strip().lower()
    if key not in exact_names:
        raise ValueError(
            f'Unsupported antibiotic {antibiotic!r}. Available calibrated models: '
            f'{sorted(models)}'
        )
    antibiotic_name = exact_names[key]
    required_features = features_by_antibiotic[antibiotic_name]

    if isinstance(feature_row, pd.DataFrame):
        if len(feature_row) != 1:
            raise ValueError('feature_row DataFrame must contain exactly one row.')
        source = feature_row.iloc[0].to_dict()
    elif isinstance(feature_row, pd.Series):
        source = feature_row.to_dict()
    elif isinstance(feature_row, dict):
        source = feature_row
    else:
        raise TypeError('feature_row must be a dict, pandas Series, or one-row DataFrame.')

    missing = [feature for feature in required_features if feature not in source]
    if missing:
        raise ValueError(
            f'Missing {len(missing)} required features. First missing features: {missing[:15]}'
        )

    ordered_values = {}
    for feature in required_features:
        value = pd.to_numeric(pd.Series([source[feature]]), errors='raise').iloc[0]
        if pd.isna(value) or not np.isfinite(float(value)):
            raise ValueError(f'Invalid value for feature {feature!r}: {source[feature]!r}')
        ordered_values[feature] = float(value)
    X = pd.DataFrame([ordered_values], columns=required_features)

    probability_resistant = float(models[antibiotic_name].predict_proba(X)[0, 1])
    threshold_info = threshold_data[antibiotic_name]
    low = float(threshold_info['susceptible_threshold'])
    high = float(threshold_info['resistant_threshold'])
    decision = make_decisions([probability_resistant], low, high)[0]
    decision_reason = 'calibrated_probability_and_no_call_thresholds'

    # The molecular-target gate is supplied by the upstream biological pipeline.
    if target_gate_passed is False:
        decision = DECISION_NO_CALL
        decision_reason = 'molecular_target_gate_failed_or_target_not_confirmed'

    if decision == DECISION_FAIL:
        confidence = probability_resistant
    elif decision == DECISION_WORK:
        confidence = 1.0 - probability_resistant
    else:
        confidence = None

    summary_match = summary[summary['antibiotic'].astype(str) == str(antibiotic_name)]
    deployment_candidate = (
        bool(summary_match.iloc[0]['deployment_candidate'])
        if not summary_match.empty and pd.notna(summary_match.iloc[0]['deployment_candidate'])
        else False
    )

    target_gate_message = (
        'passed' if target_gate_passed is True
        else 'failed' if target_gate_passed is False
        else 'not evaluated in this ML notebook; integrate the upstream drug-target gate before clinical interpretation'
    )

    return {
        'antibiotic': antibiotic_name,
        'probability_resistant': probability_resistant,
        'prediction': decision,
        'confidence': confidence,
        'decision_reason': decision_reason,
        'susceptible_threshold': low,
        'resistant_threshold': high,
        'threshold_requirement_met': bool(threshold_info['threshold_requirement_met']),
        'threshold_failure_reason': threshold_info.get('threshold_failure_reason', ''),
        'model_status': 'calibrated',
        'deployment_candidate_post_evaluation': deployment_candidate,
        'target_gate_status': target_gate_message,
        'warning': RESEARCH_WARNING,
    }

In [16]:
# 16. Example inference using one already processed hidden-test row
if calibrated_models:
    example_antibiotic = sorted(calibrated_models)[0]
    example_row = test_df[test_df['antibiotic'] == example_antibiotic].iloc[0]
    example_prediction = predict_resistance(
        example_row,
        example_antibiotic,
        target_gate_passed=None,
    )
    display(pd.DataFrame([example_prediction]))

,antibiotic,probability_resistant,prediction,confidence,decision_reason,susceptible_threshold,resistant_threshold,threshold_requirement_met,threshold_failure_reason,model_status,deployment_candidate_post_evaluation,target_gate_status,warning
0,amoxicillin,0.363663,no_call,None,calibrated_probability_and_no_call_thresholds,0.35,0.55,False,called accuracy below target,calibrated,False,not evaluated in this ML notebook; integrate t...,Research prototype. Every prediction must be c...


In [17]:
# 17. Final status and error-analysis views
calibrated_summary = summary_df[summary_df['model_status'] == 'calibrated'].copy()
unsupported_summary = summary_df[summary_df['model_status'] == 'unsupported'].copy()
deployment_summary = calibrated_summary[calibrated_summary['deployment_candidate'].fillna(False)].copy()

print('FINAL COUNTS')
print('Base models trained:', len(base_models))
print('Calibrated models:', len(calibrated_models))
print('Unsupported antibiotics:', len(unsupported_summary))
print('Fully evaluable antibiotics:', int(calibrated_summary['full_evaluation_possible'].fillna(False).sum()))
print('Post-evaluation deployment candidates:', len(deployment_summary))
print('Output directory:', OUTPUT_DIR)

print('\nCalibrated-model summary:')
display(calibrated_summary.sort_values(
    ['deployment_candidate', 'balanced_accuracy'],
    ascending=[False, False],
    na_position='last',
))

print('\nUnsupported antibiotics:')
display(unsupported_summary[['antibiotic', 'unsupported_reason', 'n_train', 'n_calibration', 'n_test']])

print('\nDeployment candidates:')
display(deployment_summary)

if not predictions_df.empty:
    false_resistant = predictions_df[
        (predictions_df['final_decision'] == DECISION_FAIL)
        & (predictions_df['is_resistant'] == 0)
    ]
    missed_resistant = predictions_df[
        (predictions_df['final_decision'] == DECISION_WORK)
        & (predictions_df['is_resistant'] == 1)
    ]
    resistant_no_calls = predictions_df[
        (predictions_df['final_decision'] == DECISION_NO_CALL)
        & (predictions_df['is_resistant'] == 1)
    ]
    no_calls = predictions_df[predictions_df['is_no_call']]

    print(f'False-resistant calls: {len(false_resistant)}')
    display(false_resistant.head(10))
    print(f'Dangerous missed-resistant calls: {len(missed_resistant)}')
    display(missed_resistant.head(10))
    print(f'Resistant no-calls: {len(resistant_no_calls)}')
    display(resistant_no_calls.head(10))
    print(f'All no-calls: {len(no_calls)}')
    display(no_calls.head(10))

FINAL COUNTS
Base models trained: 23
Calibrated models: 23
Unsupported antibiotics: 40
Fully evaluable antibiotics: 23
Post-evaluation deployment candidates: 0
Output directory: model_outputs_final

Calibrated-model summary:


,antibiotic,model_status,unsupported_reason,n_train,n_calibration,n_test,n_features,test_resistant_count,test_susceptible_count,test_resistant_prevalence,susceptible_threshold,resistant_threshold,threshold_requirement_met,threshold_failure_reason,full_evaluation_possible,evaluation_warning,deployment_candidate,tn,fp,fn,tp,balanced_accuracy,resistant_recall,susceptible_recall,resistant_precision,f1_resistant,auroc,pr_auc,brier_score,likely_to_work_count,likely_to_fail_count,no_call_count,coverage,no_call_rate,called_accuracy,called_balanced_accuracy,resistant_call_recall,susceptible_call_recall,precision_likely_to_fail,precision_likely_to_work,n_called,n_called_true_resistant,n_called_true_susceptible,balanced_accuracy_ci_lower,balanced_accuracy_ci_upper,resistant_recall_ci_lower,resistant_recall_ci_upper,susceptible_recall_ci_lower,susceptible_recall_ci_upper,f1_resistant_ci_lower,f1_resistant_ci_upper,auroc_ci_lower,auroc_ci_upper,pr_auc_ci_lower,pr_auc_ci_upper,brier_score_ci_lower,brier_score_ci_upper,coverage_ci_lower,coverage_ci_upper,called_accuracy_ci_lower,called_accuracy_ci_upper,bootstrap_method,bootstrap_iterations
1,amoxicillin,calibrated,,44,23,17,32.0,9.0,8.0,0.529412,0.35,0.55,False,called accuracy below target,True,,False,8.0,0.0,3.0,6.0,0.833333,0.666667,1.000000,1.000000,0.800000,0.909722,0.906746,0.152783,3.0,6.0,8.0,0.529412,0.470588,1.000000,1.000000,0.666667,0.375000,1.000000,1.000000,9.0,6.0,3.0,0.646250,1.000000,0.292500,1.000000,1.000000,1.000000,0.452564,1.000000,0.764489,1.000000,0.724062,1.000000,0.097157,0.222007,0.294118,0.764706,1.000000,1.000000,group-level,500.0
26,ciprofloxacin,calibrated,,626,266,262,103.0,59.0,203.0,0.225191,0.15,0.65,True,,True,,False,198.0,5.0,32.0,27.0,0.716498,0.457627,0.975369,0.843750,0.593407,0.793855,0.689059,0.109922,176.0,24.0,62.0,0.763359,0.236641,0.900000,0.770645,0.389831,0.773399,0.958333,0.892045,200.0,42.0,158.0,0.654117,0.777066,0.336932,0.572658,0.952032,0.990521,0.468984,0.697935,0.718754,0.866920,0.580176,0.790214,0.081995,0.136361,0.710632,0.816460,0.853082,0.944027,group-level,500.0
24,cephalothin,calibrated,,21,11,8,32.0,5.0,3.0,0.625000,0.45,0.55,False,called accuracy below target,True,,False,1.0,2.0,0.0,5.0,0.666667,1.000000,0.333333,0.714286,0.833333,0.466667,0.676190,0.225126,1.0,6.0,1.0,0.875000,0.125000,0.714286,0.666667,0.800000,0.333333,0.666667,1.000000,7.0,4.0,3.0,0.500000,1.000000,1.000000,1.000000,0.000000,1.000000,0.500000,1.000000,0.000000,1.000000,0.308036,1.000000,0.165434,0.300821,0.625000,1.000000,0.333333,1.000000,group-level,500.0
61,trimethoprim/sulfamethoxazole,calibrated,,258,107,90,79.0,26.0,64.0,0.288889,0.15,0.65,False,called accuracy below target; coverage below m...,True,,False,63.0,1.0,18.0,8.0,0.646034,0.307692,0.984375,0.888889,0.457143,0.750000,0.631574,0.160713,12.0,6.0,72.0,0.200000,0.800000,0.888889,0.875000,0.192308,0.171875,0.833333,0.916667,18.0,6.0,12.0,0.552433,0.742192,0.122375,0.500000,0.949555,1.000000,0.210406,0.647893,0.624070,0.855397,0.448969,0.773135,0.122097,0.204162,0.122222,0.277778,0.741250,1.000000,group-level,500.0
2,amoxicillin/clavulanic acid,calibrated,,186,70,70,92.0,25.0,45.0,0.357143,0.15,0.55,False,called accuracy below target; coverage below m...,True,,False,44.0,1.0,19.0,6.0,0.608889,0.240000,0.977778,0.857143,0.375000,0.509778,0.506939,0.234211,15.0,5.0,50.0,0.285714,0.714286,0.650000,0.650000,0.160000,0.200000,0.800000,0.600000,20.0,10.0,10.0,0.528306,0.690086,0.083333,0.408291,0.927657,1.000000,0.142857,0.561270,0.360882,0.663770,0.340112,0.668218,0.176100,0.307427,0.185714,0.385714,0.421053,0.866667,group-level,500.0
3,ampicillin,calibrated,,557,232,210,125.0,119.0,91.0,0.566667,0.25,0.85,False,called accuracy below target; coverage below m...,True,,False,37.0,54.0,23.0,96.0,0.606658,0.806723,0.406593,0.640000,0.713755,0.683073,0.749447,0.219090,2.0,11.0,197.0,0.061905,0.938095,1.000000,1.000000,0.092437,0.021978,1.000000,1.000000,13.0,11.0,2.0,0.542885,0.673686,0.728775,0.883541,0.312


Unsupported antibiotics:


,antibiotic,unsupported_reason,n_train,n_calibration,n_test
0,amikacin,insufficient class diversity in calibration,186,39,46
4,ampicillin/sulbactam,insufficient class diversity in calibration,33,4,5
5,azithromycin,insufficient class diversity in calibration,77,3,7
7,beta-lactam,insufficient class diversity in training; insu...,0,1,0
8,carbenicillin,insufficient class diversity in training; insu...,0,1,1
9,cefalotin,insufficient class diversity in calibration,18,3,8
10,cefazolin,insufficient class diversity in calibration,22,4,8
13,cefotaxime/clavulanic acid,insufficient class diversity in training; insu...,0,0,1
14,cefotetan,insufficient class diversity in training; insu...,12,1,2
16,cefpodoxime,insufficient class diversity in calibration,7,3,3



Deployment candidates:


...

,genome_id,antibiotic,is_resistant,probability_resistant,binary_prediction_at_0_5,final_decision,is_no_call,is_correct_if_called,susceptible_threshold,resistant_threshold,threshold_requirement_met,model_status,deployment_candidate,cluster_id
19,562.96565,amoxicillin/clavulanic acid,1,0.133501,0,likely_to_work,False,False,0.15,0.55,False,calibrated,False,1476
50,562.96695,amoxicillin/clavulanic acid,1,0.034652,0,likely_to_work,False,False,0.15,0.55,False,calibrated,False,1519
62,562.65565,amoxicillin/clavulanic acid,1,0.023143,0,likely_to_work,False,False,0.15,0.55,False,calibrated,False,1306
64,562.57377,amoxicillin/clavulanic acid,1,0.084022,0,likely_to_work,False,False,0.15,0.55,False,calibrated,False,1167
77,562.96696,amoxicillin/clavulanic acid,1,0.105407,0,likely_to_work,False,False,0.15,0.55,False,calibrated,False,1520
80,562.22707,amoxicillin/clavulanic acid,1,0.128516,0,likely_to_work,False,False,0.15,0.55,False,calibrated,False,632
305,562.42867,aztreonam,1,0.210830,0,likely_to_work,False,False,0.25,0.55,False,calibrated,False,1003
314,562.140887,aztreonam,1,0.201028,0,likely_to_work,False,False,0.25,0.55,False,calibrated,False,375
351,562.97224,cefotaxime,1,0.064802,0,likely_to_work,False,False,0.15,0.55,True,calibrated,False,1652
355,562.56835,cefotaxime,1,0.101621,0,likely_to_work,False,False,0.15,0.55,True,calibrated,False,1073


Resistant no-calls: 248


,genome_id,antibiotic,is_resistant,probability_resistant,binary_prediction_at_0_5,final_decision,is_no_call,is_correct_if_called,susceptible_threshold,resistant_threshold,threshold_requirement_met,model_status,deployment_candidate,cluster_id
1,562.23376,amoxicillin,1,0.363628,0,no_call,True,NaN,0.35,0.55,False,calibrated,False,785
12,562.22855,amoxicillin,1,0.363628,0,no_call,True,NaN,0.35,0.55,False,calibrated,False,672
16,562.22893,amoxicillin,1,0.363663,0,no_call,True,NaN,0.35,0.55,False,calibrated,False,686
17,562.144228,amoxicillin/clavulanic acid,1,0.276250,0,no_call,True,NaN,0.15,0.55,False,calibrated,False,416
22,562.23721,amoxicillin/clavulanic acid,1,0.204359,0,no_call,True,NaN,0.15,0.55,False,calibrated,False,865
23,562.96605,amoxicillin/clavulanic acid,1,0.188416,0,no_call,True,NaN,0.15,0.55,False,calibrated,False,1486
30,562.23035,amoxicillin/clavulanic acid,1,0.188416,0,no_call,True,NaN,0.15,0.55,False,calibrated,False,714
34,562.145413,amoxicillin/clavulanic acid,1,0.276250,0,no_call,True,NaN,0.15,0.55,False,calibrated,False,515
48,562.23311,amoxicillin/clavulanic acid,1,0.513517,1,no_call,True,NaN,0.15,0.55,False,calibrated,False,771
49,562.23652,amoxicillin/clavulanic acid,1,0.276250,0,no_call,True,NaN,0.15,0.55,False,calibrated,False,848


All no-calls: 672


,genome_id,antibiotic,is_resistant,probability_resistant,binary_prediction_at_0_5,final_decision,is_no_call,is_correct_if_called,susceptible_threshold,resistant_threshold,threshold_requirement_met,model_status,deployment_candidate,cluster_id
0,562.23490,amoxicillin,0,0.363663,0,no_call,True,NaN,0.35,0.55,False,calibrated,False,812
1,562.23376,amoxicillin,1,0.363628,0,no_call,True,NaN,0.35,0.55,False,calibrated,False,785
5,562.23310,amoxicillin,0,0.363628,0,no_call,True,NaN,0.35,0.55,False,calibrated,False,770
6,562.22868,amoxicillin,0,0.363628,0,no_call,True,NaN,0.35,0.55,False,calibrated,False,678
9,562.22951,amoxicillin,0,0.363628,0,no_call,True,NaN,0.35,0.55,False,calibrated,False,699
12,562.22855,amoxicillin,1,0.363628,0,no_call,True,NaN,0.35,0.55,False,calibrated,False,672
15,562.23652,amoxicillin,0,0.363628,0,no_call,True,NaN,0.35,0.55,False,calibrated,False,848
16,562.22893,amoxicillin,1,0.363663,0,no_call,True,NaN,0.35,0.55,False,calibrated,False,686
17,562.144228,amoxicillin/clavulanic acid,1,0.276250,0,no_call,True,NaN,0.15,0.55,False,calibrated,False,416
20,562.22985,amoxicillin/clavulanic acid,0,0.267801,0,no_call,True,NaN,0.15,0.55,False,calibrated,False,704
